# GRAST WP07.1 — Training engine smoke trên Colab

Notebook này chỉ kiểm tra engine/AMP/checkpoint trong vài optimizer updates. Chưa chạy corpus pretraining hoặc CoT SFT thật; các runner dài sẽ đến ở WP07.3/4.

Chọn **Runtime → Change runtime type → GPU**, ưu tiên A100. Mọi lệnh đều in log trực tiếp; cell train dùng unbuffered Python và chuyển từng dòng stdout/stderr ra notebook. Không có quiet mode.


In [ ]:
import subprocess
import sys
from pathlib import Path

SOURCE_REF = "feat/wp07-1-training-engine"  # Nhánh WP07.1; commit thực tế được in ở dưới.
PROJECT_ROOT = Path("/content/GRAST-WP07-engine")
OUTPUT_ROOT = Path("/content/grast-wp07-engine-runs")


def run_logged(command, *, cwd=None):
    print("RUN:", " ".join(map(str, command)), flush=True)
    with subprocess.Popen(
        list(map(str, command)),
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)


if not PROJECT_ROOT.exists():
    run_logged(
        [
            "git",
            "clone",
            "--branch",
            SOURCE_REF,
            "https://github.com/CuongBien/GRAST.git",
            PROJECT_ROOT,
        ]
    )
run_logged(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT)
run_logged([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], cwd=PROJECT_ROOT)

In [ ]:
import torch

print("Python:", sys.version, flush=True)
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda, flush=True)
if not torch.cuda.is_available():
    raise RuntimeError("Bật GPU runtime trước khi chạy smoke này; không tự fallback CPU.")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
PRECISION = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
print("Smoke precision:", PRECISION, flush=True)

## Chạy smoke ngắn

Sáu optimizer updates trên model nhỏ, save ở step 2 rồi resume và so toàn bộ model/optimizer/scheduler/scaler. FP16 overflow được in cùng loss scale, không tăng scheduler khi update bị skip. CSV loss ở đây là **diagnostic eval trên cùng fixture**, không phải validation curve của corpus.


In [ ]:
run_logged(
    [
        sys.executable,
        "-u",
        "-m",
        "llm_scratch.research.training_acceptance",
        "--device",
        "cuda",
        "--precision",
        PRECISION,
        "--output-root",
        OUTPUT_ROOT,
    ],
    cwd=PROJECT_ROOT,
)

In [ ]:
import json

runs = sorted(OUTPUT_ROOT.glob("*wp07-1-training-engine*"))
if not runs:
    raise RuntimeError("Không có run directory.")
RUN_DIR = runs[-1]
report_path = RUN_DIR / "artifacts/training_engine_report.json"
print("Run:", RUN_DIR, flush=True)
print(report_path.read_text(), flush=True)
report = json.loads(report_path.read_text())
assert report["status"] == "passed"
assert report["checks"]["exact_resume_model_optimizer_scheduler_scaler"]
print("wp07_1_colab_smoke_passed", flush=True)

## Lưu artifacts trước khi tắt runtime

Bundle gồm log, metadata, config, loss history và checkpoint của smoke model nhỏ. Đây chưa phải checkpoint pretraining baseline. Khi chạy corpus ở các phase sau, checkpoint cần lưu định kỳ vào Drive và log vẫn hiển thị trong notebook.


In [ ]:
import shutil

from google.colab import files

bundle = shutil.make_archive(
    "/content/grast-wp07-engine-smoke", "zip", root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name
)
print("Artifact bundle:", bundle, flush=True)
files.download(bundle)